# Phase 3, orientation — the whole agent, one call at a time

Every other Phase-3 notebook builds *one* new piece of the harness. This one
builds nothing. It exists to answer a question that the other notebooks
assume you already have an answer to:

**What can this agent actually do, and what happens on disk when it does it?**

The plan is to make exactly one tool call per section, and after each one look
at three things:

1. the arguments the model chose,
2. the summary it gets back,
3. the files that appeared in the run directory.

Nothing is automated until the very last section. By the time an LLM takes over,
you will have done its whole job by hand.

## 1. The three moving parts

The harness is smaller than it looks. There are three pieces and it is worth
naming them before touching anything.

| Part | File | What it is |
|---|---|---|
| **The loop** | `harness/agent.py` | A PydanticAI agent. Declares which functions the model may call, picks a model, and nothing else. |
| **The tools** | `harness/tools.py` | Plain Python functions. No LLM anywhere in them. *All* the behaviour lives here. |
| **The run dir** | `runctx.py` (10 lines) | A folder. Every artifact goes in it. This folder **is** the state of the run. |

The split matters because of what it buys: the tools are unit-testable without
an API key, and a human can drive the entire pipeline by hand — which is what
the rest of this notebook does.

In [1]:
import inspect as pyinspect

from tabular_agent.harness import agent as agent_mod

# The tool surface is whatever is decorated @agent.tool inside build_agent.
# Read it off the source rather than trusting a list in a docstring.
lines = pyinspect.getsource(agent_mod.build_agent).splitlines()
tool_names = [
    lines[i + 1].split("def ")[1].split("(")[0]
    for i, line in enumerate(lines)
    if line.strip() == "@agent.tool"
]
print(f"{len(tool_names)} tools:")
for name in tool_names:
    print("  -", name)

6 tools:
  - inspect_data
  - fit_encoding
  - transform_features
  - train_tabular_model
  - explain_table
  - finalize_run


That list is the entire reachable surface. There is no shell, no file read, no
free-form query, no way to run code except through two of those five. If a
capability is not in that list, the agent cannot do it — not "is discouraged
from doing it".

Two of the five take something open-ended: `transform_features` takes Python,
and `fit_encoding` takes a column list. The other three are a fixed menu.

### The contract every tool obeys

Each one takes **paths plus typed parameters** and returns a **small JSON-able
summary plus paths**. Never a DataFrame.

Two consequences, and they are the reason the harness is shaped this way:

- **Context grows slowly.** A run can span dozens of cycles without the
  conversation filling up, because no tool result is ever more than a few
  hundred tokens.
- **The state is on disk, not in the conversation.** A crash or a context reset
  loses nothing: hand a fresh process the same run directory and it continues.

The flip side, which becomes important later: **the model never sees a row of
data.** It reasons entirely from summaries. Keep that in mind — it is the
constraint the whole design is built around, and the one that the residual-
feedback work is eventually about relaxing safely.

## 2. A dataset with a rule we know

Same synthetic apartments table as the step-2 and step-3 notebooks, so the
numbers here are comparable to those.

In [2]:
import json
import os
import tempfile

import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 800

area = rng.uniform(30, 160, size=n).round(1)
price_per_m2 = rng.uniform(2000, 9000, size=n).round(0)
price = (area * price_per_m2).round(0)
year_built = rng.integers(1900, 2024, size=n)
rooms = np.clip((area / 30 + rng.normal(0, 0.5, size=n)).round(), 1, 6)
city = rng.choice(["berlin", "hamburg", "koeln"], size=n)
listed_at = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    rng.integers(0, 730, size=n), unit="D"
)

# The real rule: season dominates, weekend listings do well, cheap-per-m2 and
# not-ancient help. Only the last two are visible in the raw numeric columns.
season = np.where(
    np.isin(listed_at.month, [3, 4, 5]),
    1.6,
    np.where(np.isin(listed_at.month, [11, 12, 1]), -1.4, 0.0),
)
weekend = np.where(listed_at.dayofweek >= 5, 0.9, 0.0)
logit = (
    season
    + weekend
    - (price_per_m2 - 5500) / 1500
    - ((2026 - year_built) - 60) / 90
    + rng.normal(0, 0.6, size=n)
)

df = pd.DataFrame({
    "price": price,
    "area": area,
    "rooms": rooms,
    "year_built": year_built,
    "city": city,
    "listed_at": listed_at.strftime("%Y-%m-%d"),
    "sold_fast": (logit > 0).astype(int),
})

workdir = tempfile.mkdtemp(prefix="phase3_map_")
data_path = os.path.join(workdir, "apartments.csv")
df.to_csv(data_path, index=False)
df.head()

,price,area,rooms,year_built,city,listed_at,sold_fast
0,999585.0,111.3,4.0,1990,berlin,2024-03-04,0
1,454020.0,146.6,6.0,1994,hamburg,2024-02-18,1
2,1005852.0,130.8,4.0,1976,berlin,2023-04-16,1
3,289265.0,59.3,1.0,1963,hamburg,2023-11-03,0
4,516327.0,69.0,2.0,2005,berlin,2023-10-07,0


Worth knowing what we planted, because we will watch the agent's tools discover
it:

- The strongest driver is **the month in `listed_at`** — and `listed_at` is a
  string, so no model can use it until something derives a month from it.
- `city` is a string too, and (deliberately) carries almost no signal. It is
  there so we have a categorical column to encode.
- `price_per_m2` drives the outcome but is **not a column** — only `price` and
  `area` are. A ratio has to be built, *or* inferred by the model from the two
  columns it already has. Section 10 finds out which.

So the run needs both open-ended tools: `transform_features` for the date parts,
`fit_encoding` for `city`.

## 3. The run directory, before anything happens

`RunSession` is the harness-side handle on one run. It holds three things: where
to write, which raw table, and the task spec. That is all — no cached state, no
loaded data.

In [3]:
from tabular_agent.harness.session import RunSession
from tabular_agent.taskspec import TaskSpec

session = RunSession(
    run_dir=os.path.join(workdir, "run"),
    data_path=data_path,
    spec=TaskSpec(target="sold_fast", n_splits=4),
)

print("run_dir  :", session.run_dir)
print("data_path:", session.data_path)
print("target   :", session.spec.target)

run_dir  : /var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/phase3_map_7lg2epko/run
data_path: /var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/phase3_map_7lg2epko/apartments.csv
target   : sold_fast


Here is the helper we will use after every single call. It prints the run
directory as a tree with file sizes — the point being to watch it fill up.

In [4]:
def tree(root, label=None):
    """Print `root` as an indented tree with file sizes. Our x-ray of the run dir."""
    root = os.path.abspath(root)
    print(label or os.path.basename(root) + "/")
    entries = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        rel = os.path.relpath(dirpath, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if rel != ".":
            entries.append(("  " * depth + os.path.basename(dirpath) + "/", None))
        for name in sorted(filenames):
            size = os.path.getsize(os.path.join(dirpath, name))
            entries.append(("  " * (depth + 1) + name, size))
    if not entries:
        print("  (empty)")
    for line, size in entries:
        print(line if size is None else f"{line:<44} {size:>9,} b")


tree(session.run_dir)

run/
  (empty)


Empty. `RunSession.__init__` creates the directory and writes nothing into it.
Even the journal file does not exist yet — `journal.load` returns a default dict
when there is no file, so "no run yet" and "empty run" are the same thing.

**Check yourself:** the session knows `data_path` and `target`, but not the task
type or the metric. Why are those not arguments?

> Because they are derivable from the target column, and asking a human to state
> them is asking them to get them wrong. `session.resolved()` auto-detects on
> first use and then **caches** — the comment at `session.py:44` is the reason:
> if detection re-ran per call and ever gave a different answer, the leaderboard
> would be comparing scores measured with different metrics, silently.

## 4. Call 1 — `inspect_data`

The mandatory first call. It profiles the table and pins down the task type and
metric that every later score will be measured with.

In [5]:
from tabular_agent.harness.tools import inspect_data

profile = inspect_data(session)

print("task_type:", profile["task_type"])
print("metric   :", profile["metric"])
print("n_splits :", profile["n_splits"])
print("shape    :", profile["n_rows"], "x", profile["n_cols"])
print("unusable :", profile["non_numeric_columns"])
print()
print(profile["note"])

task_type: binary
metric   : logloss
n_splits : 4
shape    : 800 x 7
unusable : ['city', 'listed_at']

Non-numeric columns are not usable by the models as-is; encode them with fit_encoding to bring them in — method='ordinal' (stating the level order) when the categories are genuinely ordered, method='frequency' or 'target' when they are nominal.


`non_numeric_columns` is the interesting field. The models are GBDTs fed a
numeric matrix, so `city` and `listed_at` would crash the fit. The tool that
trains does not cast them — it **drops them and says so**.

That is a deliberate choice over a convenient one: an ad-hoc cast (say,
alphabetical codes for `city`) would work and would quietly skew train against
serve, because nothing would have stored the mapping. The honest move is to
refuse and point at `fit_encoding`, which fits per fold and persists what it
learned.

In [6]:
# The per-column profile, capped at MAX_REPORTED_COLUMNS (60) so a wide table
# cannot blow up the model's context.
pd.DataFrame(profile["columns"]).T

,dtype,n_missing,n_unique,categories
price,float64,0,800,NaN
area,float64,0,587,NaN
rooms,float64,0,6,NaN
year_built,int64,0,124,NaN
city,str,0,3,"[berlin, hamburg, koeln]"
listed_at,str,0,486,NaN
sold_fast,int64,0,2,NaN


Now the disk. This is the first call, so this is the first thing written.

In [7]:
tree(session.run_dir)

run/
  run_journal.json                                 265 b


In [8]:
# One file, and only one key in it so far.
journal = json.load(open(os.path.join(session.run_dir, "run_journal.json")))
print(json.dumps({k: v for k, v in journal.items() if v not in ([], None)}, indent=2))

{
  "schema_version": 1,
  "task": {
    "target": "sold_fast",
    "task_type": "binary",
    "metric": "logloss"
  },
  "stop": {
    "refusals": 0,
    "finalized": false
  }
}


So `inspect_data` writes **the task block and nothing else**. It creates no
table, no experiment. It is pure orientation, plus the one act of recording that
makes every later score comparable.

Note the empty containers that already exist in the schema: `tables`,
`leaderboard`, `log`, and `stop`. The journal has its final shape from the first
write.

## 5. Call 2 — the baseline

`train_tabular_model(model_type="baseline")`, before any real model. The system
prompt makes this step 2 of the workflow and the reason is blunt: a model that
cannot beat the trivial predictor is a bug, not a model, and you cannot tell
without the number.

In [9]:
from tabular_agent.harness.tools import train_tabular_model

e1 = train_tabular_model(session, model_type="baseline")
print(json.dumps(e1, indent=2))

{
  "trained": true,
  "experiment_id": "e1",
  "cv_score": 0.6903418151196927,
  "fold_spread": 0.00037570679522003214,
  "model_type": "baseline",
  "n_features": 4,
  "source_table": "/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/phase3_map_7lg2epko/apartments.csv",
  "dropped_non_numeric": [
    "city",
    "listed_at"
  ],
  "vs_baseline": null,
  "vs_source_table": null,
  "columns_by_table": [
    {
      "table": "the raw input table",
      "method": null,
      "n_columns": 4
    }
  ],
  "artifact_path": "experiments/e1",
  "note": "That is the best result so far. 24 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.",
  "vs_source_basis": "the raw input table was not derived from another table"
}


Read the fields in three groups, because this return value is doing three jobs
at once:

**The measurement** — `cv_score`, `fold_metrics`, `fold_spread`. The score is
the mean across the 4 folds; the spread is their standard deviation. `metric`
and `higher_is_better` come along so the model never has to guess the direction.

**The bookkeeping** — `experiment_id`, `artifact_path`, `source_table`,
`n_features`, `dropped_non_numeric`. Note `dropped_non_numeric` repeats the
warning from `inspect_data`: the columns are still being left on the table.

**The steering** — `since_improvement`, `since_improvement_on_table`,
`experiments_remaining`, and `note`. Every training result tells the model where
the run stands, in English. That is the advisory half of the stop rule, and it
exists so a cooperative model never has to be refused at all.

In [10]:
tree(session.run_dir)

run/
  run_journal.json                               1,165 b
  experiments/
    e1/
      final_model.joblib                           435 b
      meta.json                                    448 b
      oof.parquet                                1,317 b


An experiment is a **directory**, not a row. Three files:

- `meta.json` — the full record: model type, params, metric, cv_score,
  fold_metrics, feature_names, source_table.
- `final_model.joblib` — the model refit on **all** rows. The per-fold models
  are used to score and then discarded; this is the one that would ship.
- `oof.parquet` — the out-of-fold predictions, one row per training row.

`oof.parquet` is worth pausing on, because it is the file that matters for
what comes next. It is written on every single training call and **nothing ever
reads it**. It holds exactly the information a human would look at to decide
what feature to build next, and no tool exposes it. That gap is the
residual-feedback work.

In [11]:
exp_dir = os.path.join(session.run_dir, e1["artifact_path"])
print(json.dumps(json.load(open(os.path.join(exp_dir, "meta.json"))), indent=2))

{
  "id": "e1",
  "model_type": "baseline",
  "params": {},
  "task_type": "binary",
  "metric": "logloss",
  "cv_score": 0.6903418151196927,
  "fold_metrics": [
    0.6907175219149128,
    0.6907175219149128,
    0.6899661083244727,
    0.6899661083244727
  ],
  "feature_names": [
    "price",
    "area",
    "rooms",
    "year_built"
  ],
  "source_table": "/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/phase3_map_7lg2epko/apartments.csv"
}


In [12]:
oof = pd.read_parquet(os.path.join(exp_dir, "oof.parquet"))
print("shape:", oof.shape, "| rows in the raw table:", len(df))
oof.head()

shape: (800, 1) | rows in the raw table: 800


,oof_0
0,0.536667
1,0.536667
2,0.538333
3,0.538333
4,0.538333


Same number of rows as the table, in the same order — `train_cv` resets the
index and assigns `oof[va] = preds` per fold, so **row *i* of `oof.parquet` is
row *i* of the source table**. The alignment is positional, and it is the only
join key you would need to look at residuals by slice.

The baseline predicts one constant (the training-fold mean), which is why every
value is the same to three decimals.

## 6. Call 3 — a real model on the raw table

Same tool, different `model_type`. Nothing else changes.

In [13]:
e2 = train_tabular_model(session, model_type="lgbm")

print("cv_score   :", round(e2["cv_score"], 4))
print("fold_spread:", round(e2["fold_spread"], 4))
print("vs_baseline:", e2["vs_baseline"])
print("n_features :", e2["n_features"], "of", df.shape[1] - 1, "feature columns")
print("dropped    :", e2["dropped_non_numeric"])
print()
print(e2["note"])

cv_score   : 0.6881
fold_spread: 0.0416
vs_baseline: 0.002229
n_features : 4 of 6 feature columns
dropped    : ['city', 'listed_at']

That is the best result so far. 23 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.


Now look at those numbers properly, because they are more interesting than a
success would have been.

`vs_baseline` is about `-0.002` and lower is better for logloss, so LightGBM
beat the trivial predictor. By 0.002 — while its own score moves by **0.04**
between folds. The gain is an order of magnitude smaller than the noise in the
measurement it is made of. A gradient-boosted machine on real numeric columns
has essentially tied with predicting the mean.

Which is exactly right, and it is why the dataset was built this way: the
strongest driver is the month, and the month is trapped inside a string column
that got dropped. **This is the state the agent starts every run in** — the
score is not limited by the model, it is limited by the features, and the only
evidence of that is `dropped_non_numeric` sitting there in the result.

Two new fields appear now that there is a real model with feature importances:

In [14]:
print("columns_by_table:")
print(json.dumps(e2["columns_by_table"], indent=2))
print()
print("vs_source_table:", e2["vs_source_table"])
print("vs_source_basis:", e2["vs_source_basis"])


columns_by_table:
[
  {
    "table": "the raw input table",
    "method": null,
    "n_columns": 4
  }
]

vs_source_table: None
vs_source_basis: the raw input table was not derived from another table


One entry, `"the raw input table"` — there are no derived columns yet, so there
is nothing to attribute.

`vs_source_table` is `None` and that is not a missing value. The raw table was
not derived from anything, so a gain over its source does not exist —
`vs_source_basis` says exactly that rather than leaving a bare `null`. It becomes
a number in section 8.

**Check yourself:** credit is reported per *table* — which is per tool call, since
one call writes one table — never per column. Why not per column, when that is
obviously more detailed?

> Because a column is finer than any move the agent can make: it cannot add half a
> tool call. Worse, per-column feedback misleads — a feature that only pays off
> next to another scores nothing on its own, so reporting per column is how an
> interaction partner gets dropped before its partner is ever built. The call is
> the unit of *decision*, so it is the unit of credit.


## 7. Call 4 — `transform_features`, the escape hatch

Now the first of the two open-ended tools. The agent writes pandas, we run it in
a sandbox, and the result becomes a **new table**.

The one idea in this call: get the date out of `listed_at`, since we know the
month is the strongest signal in the data.

In [15]:
from tabular_agent.harness.tools import transform_features

date_code = '''
import pandas as pd
def transform_features(df):
    df = df.copy()
    ts = pd.to_datetime(df["listed_at"])
    df["month"] = ts.dt.month
    df["dayofweek"] = ts.dt.dayofweek
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
    return df.drop(columns=["listed_at"])
'''

t1 = transform_features(session, date_code)
print(json.dumps(t1, indent=2))

{
  "table": "tables/t1.parquet",
  "new_columns": [
    "month",
    "dayofweek",
    "is_weekend"
  ],
  "n_rows": 800,
  "n_cols": 9,
  "source_table": "/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn/T/phase3_map_7lg2epko/apartments.csv",
  "non_numeric_columns": [
    "city"
  ],
  "note": "Pass this table to train_tabular_model to use the new columns. The code is stored in the table's lineage and re-runs at predict time; the target column was not visible to it."
}


`new_columns` lists what appeared; the returned `table` path is what you pass to
train on it. Note `non_numeric_columns` is now down to `["city"]` — the string
date is gone, replaced by three integers.

Now the disk, which is where this tool gets interesting.

In [16]:
tree(session.run_dir)

run/
  run_journal.json                               2,525 b
  experiments/
    e1/
      final_model.joblib                           435 b
      meta.json                                    448 b
      oof.parquet                                1,317 b
    e2/
      final_model.joblib                       680,420 b
      meta.json                                    443 b
      oof.parquet                                8,610 b
  tables/
    t1.parquet                                  16,907 b
    t1.pipeline.json                               438 b


`transform_features` writes **two** files, and the second one is the design.

In [17]:
lineage = json.load(open(os.path.join(session.run_dir, "tables", "t1.pipeline.json")))
print(json.dumps(lineage, indent=2))

{
  "task_type": "binary",
  "steps": [
    {
      "type": "transform",
      "code": "\nimport pandas as pd\ndef transform_features(df):\n    df = df.copy()\n    ts = pd.to_datetime(df[\"listed_at\"])\n    df[\"month\"] = ts.dt.month\n    df[\"dayofweek\"] = ts.dt.dayofweek\n    df[\"is_weekend\"] = (ts.dt.dayofweek >= 5).astype(int)\n    return df.drop(columns=[\"listed_at\"])\n",
      "func_name": "transform_features"
    }
  ]
}


The sidecar stores **the code, not the columns**.

That is the whole trick for making a derived feature safe to serve. The columns
in `t1.parquet` are for training on. The *recipe* is what gets replayed at
predict time — so a stored feature can never drift from the code that made it,
because the feature is not stored at all.

It also means the pipeline is a chain: build a table on top of `t1` and its
sidecar will hold two steps, in order.

**Check yourself:** the frame handed to that code did **not** contain
`sold_fast`. Two independent reasons were given for dropping it. Which one is
load-bearing?

> Leakage is the obvious one — a column cannot be a function of the target if
> the target was never there. But the deciding reason is that **there is no
> target at predict time**. Since only the code is stored and re-run, code
> referencing `sold_fast` would either crash at serve or silently compute
> something different. Even a *non-leaky* glance at the target would break the
> replay, which is why this is enforced rather than advised.

In [18]:
# What actually landed in the table. Same rows, three new columns, no listed_at,
# and the target carried through untouched.
t1_df = pd.read_parquet(os.path.join(session.run_dir, t1["table"]))
print("columns:", list(t1_df.columns))
t1_df.head(3)

columns: ['price', 'area', 'rooms', 'year_built', 'city', 'month', 'dayofweek', 'is_weekend', 'sold_fast']


,price,area,rooms,year_built,city,month,dayofweek,is_weekend,sold_fast
0,999585.0,111.3,4.0,1990,berlin,3,0,0,0
1,454020.0,146.6,6.0,1994,hamburg,2,6,1,1
2,1005852.0,130.8,4.0,1976,berlin,4,6,1,1


## 8. Call 5 — train on the new table

The `table` argument is how the agent picks what to train on. Pass the path the
previous call handed back.

In [19]:
e3 = train_tabular_model(session, table=t1["table"], model_type="lgbm")

print("cv_score       :", round(e3["cv_score"], 4), "  (was", round(e2["cv_score"], 4), "on raw)")
print("vs_baseline    :", e3["vs_baseline"])
print("vs_source_table:", e3["vs_source_table"])
print("source_table   :", e3["source_table"])
print()
print(json.dumps(e3["columns_by_table"], indent=2))


cv_score       : 0.4975   (was 0.6881 on raw)
vs_baseline    : 0.192799
vs_source_table: 0.190571
source_table   : tables/t1.parquet

[
  {
    "table": "the raw input table",
    "method": null,
    "n_columns": 4
  },
  {
    "table": "tables/t1.parquet",
    "method": "transform",
    "n_columns": 3
  }
]


`0.688` to `0.498`. That is the whole game in one call — not a new model, not a
tuned hyper-parameter, just three integers pulled out of a string.

`vs_source_table` is finally a number, because `t1` was built from the raw table
and the raw table *was* trained on, in section 6 — with the same model and the
same params, which is the condition. That conditionality is deliberate: the
marginal number exists only if the previous step was measured the same way, so
measuring it **pays off** rather than being a rule imposed on the agent.

But `columns_by_table` is only structure — which call put which columns here. It
carries no score, because the question "which of these two calls earned the
`0.19`?" cannot be answered from a fitted model. It takes refits.


In [20]:
# Which of those two calls earned the gain? That takes refits, so it is its own tool.
from tabular_agent.harness.tools import explain_table

credit = explain_table(session, table=t1["table"], model_type="lgbm")
for r in credit["credit_by_table"]:
    print(f"  {r['table']:24} n={r['n_columns']:2} contribution={r['contribution']}")


  the raw input table      n= 4 contribution=0.12442
  tables/t1.parquet        n= 3 contribution=0.190571


```
  the raw input table      n= 4 contribution=0.12442
  tables/t1.parquet        n= 3 contribution=0.190571
```

Hold out the three date columns and the score loses `0.191`; hold out the four
original columns and it loses `0.124`. Both calls are carrying real weight.

That second number is worth sitting with, because section 6 showed those same four
columns **tying the baseline** on their own. Useless alone, worth `0.124` here.
Nothing is contradictory: `price`, `area`, `rooms` and `year_built` say little
about the target until the model can also see *when* the listing went up, and then
they say a lot. That is an interaction, and it is exactly the case per-column
feedback destroys — score those four columns alone, get ~0, drop them, and never
discover that the date features were about to make them valuable.

So three questions, three different answers, all about the same four columns:

| question | answer |
|---|---|
| do they beat the baseline alone? | no (section 6) |
| where does the fitted model spend its splits? | 77% of them, here |
| what does the score lose without them? | `0.124` |

Only the third is evidence about whether keeping them is right, and only the third
requires refitting. The `77%` is what split-count importance reports, and an
earlier version of this notebook shipped it on every training result: `price` and
`area` are continuous with hundreds of split points, so a tree spends splits there
whether or not they pay. It is a description of attention, not of value — and it
cannot tell the case above from a column that is genuinely redundant.

Leave-one-out can, which is why `explain_table` refits instead. Every number it
returns is that call's worth *given the rest*, so a `0.0` means "this table does
not need these columns", never "the idea is bad". Nothing is pruned for you.


## 9. Call 6 — `fit_encoding`, the tool that is allowed to learn

`transform_features` cannot look at the target and cannot learn a parameter from
the batch. So how does a category's target mean ever get built? Through this
tool, which is structured precisely because it is the dangerous one.

We stack it on `t1`, so the new table has the date features *and* an encoded
`city`.

In [21]:
from tabular_agent.harness.tools import fit_encoding

t2 = fit_encoding(
    session,
    columns=["city"],
    method="target",
    table=t1["table"],
)
print(json.dumps(t2, indent=2))

{
  "table": "tables/t2.parquet",
  "method": "target",
  "new_columns": [
    "city_te"
  ],
  "n_rows": 800,
  "source_table": "tables/t1.parquet",
  "non_numeric_columns": [
    "city"
  ],
  "note": "Pass this table to train_tabular_model to use the new columns. The values stored here are out-of-fold; the encoder fitted on all rows is kept in the table's lineage and is what runs at predict time."
}


Read that `note` carefully, because it describes the one genuinely subtle thing
in the harness:

- the **table** gets **out-of-fold** values — honest to train on, because each
  row's encoding was computed without that row's fold;
- the **sidecar** stores the encoder fitted on **all rows** — which is what is
  correct at predict time, where there are no folds.

Two different sets of numbers for two different jobs. That is why target
encoding is a tool and not something the agent is trusted to write.

In [22]:
lineage2 = json.load(open(os.path.join(session.run_dir, "tables", "t2.pipeline.json")))
print("steps in t2's lineage:", len(lineage2["steps"]))
for i, step in enumerate(lineage2["steps"]):
    detail = step.get("func_name") or step["encoder"]["kind"]
    print(f"  {i}: type={step['type']:<10} ({detail})")

steps in t2's lineage: 2
  0: type=transform  (transform_features)
  1: type=encode     (target)


Two steps, in order: the date transform inherited from `t1`, then the encoder.
This is the chain the bundle will eventually need in order to rebuild features
from raw input.

In [23]:
# The out-of-fold values that landed in the table, next to the category.
t2_df = pd.read_parquet(os.path.join(session.run_dir, t2["table"]))
t2_df.groupby("city")["city_te"].agg(["mean", "std", "count"])

,mean,std,count
city,,,
berlin,0.501420,0.004100,278
hamburg,0.574762,0.016434,269
koeln,0.539336,0.015718,253


Same category, *different* values within it — that spread is the out-of-fold
machinery. A single fitted mean per city would show `std = 0`.

**Check yourself:** `fit_encoding` uses the same fold seed as training
(`make_folds(..., seed=spec.seed)` in both). What breaks if it uses a different
one?

> The encoding for a row would have been computed from rows that the model
> scoring that row also trained on. The target would leak across the fold
> boundary, and the cv_score would be optimistic — the exact failure the
> out-of-fold encoding exists to prevent. Sharing the seed is what makes "held
> out" mean the same thing in both places.

In [24]:
e4 = train_tabular_model(session, table=t2["table"], model_type="lgbm")

print("cv_score       :", round(e4["cv_score"], 4))
print("fold_spread    :", round(e4["fold_spread"], 4), " <- compare this to the gain below")
print("vs_source_table:", e4["vs_source_table"], " (source is t1, trained in section 8)")
print()
print(json.dumps(e4["columns_by_table"], indent=2))

cv_score       : 0.4686
fold_spread    : 0.0807  <- compare this to the gain below
vs_source_table: 0.028904  (source is t1, trained in section 8)

[
  {
    "table": "the raw input table",
    "method": null,
    "n_columns": 4
  },
  {
    "table": "tables/t1.parquet",
    "method": "transform",
    "n_columns": 3
  },
  {
    "table": "tables/t2.parquet",
    "method": "target",
    "n_columns": 1
  }
]


Stop and read this one carefully, because we know the ground truth and the
harness does not.

`city` was generated by `rng.choice` — **independent of the target**. There is
nothing in it. And yet:

- `city_te` took about 9% of the splitting, and
- `vs_source_table` is **positive**: roughly `+0.029`, an apparent improvement.

The score got better by adding a column we know to be noise. That is not a bug in
the encoder — the out-of-fold machinery worked exactly as intended. It is what
selecting on a noisy measurement looks like. The fold spread on this experiment is
about `0.081`, so a `0.029` "gain" is roughly a third of the amount this very
model's score moves when you merely change which rows it trains on.

Now the uncomfortable part. The harness's own noise floor is
`min_gain_ratio=0.25` × the incumbent's fold spread — that is `0.020` here. So
`0.029` **clears the bar** and gets recorded as progress. The rule that exists
precisely to reject noise has just accepted some.

This is worth knowing rather than hiding: the stop rule is a defence against
*sustained* self-deception (a run that grinds out ten sub-noise gains and calls it
progress), not a per-experiment lie detector. With one comparison and a spread
this wide, no threshold can be right. The real answer to "was `city` worth it"
would be repeating the comparison across fold seeds, which nothing here does.

**Check yourself:** so why keep `vs_source_table` at all, if it can report a gain
from pure noise?

> Because the alternative on offer is not a better signal, it is *no* signal —
> which is what the agent had: one scalar per experiment, and no way to tell
> which of its ideas moved it. A noisy marginal number that the agent can weigh
> against the fold spread sitting next to it in the same result beats nothing.
> The failure to avoid is presenting it as a verdict, which is why the fold
> spread ships alongside it in every single result.

## 10. Branching — tables are immutable

Nothing so far has modified a table. Every call read one and wrote a new one. So
the run is not a line, it is a tree, and the agent can put two competing ideas
on the same base.

Let us build a *second* child of `t1` — the price ratio, the other thing we know
is in the data — and compare it against `t2` on equal footing.

In [25]:
ratio_code = '''
def transform_features(df):
    df = df.copy()
    df["price_per_m2"] = df["price"] / df["area"]
    df["area_per_room"] = df["area"] / df["rooms"]
    return df
'''

t3 = transform_features(session, ratio_code, table=t1["table"])
print("new table   :", t3["table"])
print("built on    :", t3["source_table"])
print("new columns :", t3["new_columns"])

new table   : tables/t3.parquet
built on    : tables/t1.parquet
new columns : ['price_per_m2', 'area_per_room']


In [26]:
e5 = train_tabular_model(session, table=t3["table"], model_type="lgbm")

print("t2 (city encoded) :", round(e4["cv_score"], 4), " vs_source:", e4["vs_source_table"])
print("t3 (price ratios) :", round(e5["cv_score"], 4), " vs_source:", e5["vs_source_table"])
print()
for r in explain_table(session, table=t3["table"], model_type="lgbm")["credit_by_table"]:
    print(f"  {r['table']:24} n={r['n_columns']:2} contribution={r['contribution']}")


t2 (city encoded) : 0.4686  vs_source: 0.028904
t3 (price ratios) : 0.5236  vs_source: -0.026012



  the raw input table      n= 4 contribution=0.033412
  tables/t1.parquet        n= 3 contribution=0.301366
  tables/t3.parquet        n= 2 contribution=-0.026012


Two branches off the same base, each with its own marginal number against the same
measured starting point. And the ratio branch **lost** — `vs_source_table` is
`-0.026`, so `t3` is worse than the `t1` it was built on, and `explain_table`
agrees: those two columns score `-0.026`, the only negative number in the run.

That deserves a pause, because `price_per_m2` is a real driver in the generating
rule. The model already had `price` and `area`, and a tree can approximate their
ratio by splitting on both — so the explicit column added almost no information
while costing the fit two more columns to spend splits on.

Set it against section 8 and the pair is the whole lesson in reading this feedback:

| | alone | contribution here |
|---|---|---|
| the 4 original columns (§8) | tie the baseline | **+0.124** — needed |
| the 2 ratio columns (§10) | a real driver | **-0.026** — redundant |

Both look unpromising by one measure and the truth is opposite in each case. No
single number read off a fitted model separates them; refitting without each does.

And notice what the harness did about the negative one: **nothing.** `t3` still
exists, nothing was pruned, no warning was raised. Tables are immutable, so the
agent is free to keep going from `t2`, or come back to `t3` later and add the
feature that makes the ratio pay off. Abandoning a branch costs nothing.

This is the shape of the search: **build a table, train on it, read the feedback,
build another table.** Sections 7 to 10 are the loop; everything else is setup and
teardown.


In [27]:
tree(session.run_dir)

run/
  run_journal.json                               7,027 b
  experiments/
    e1/
      final_model.joblib                           435 b
      meta.json                                    448 b
      oof.parquet                                1,317 b
    e2/
      final_model.joblib                       680,420 b
      meta.json                                    443 b
      oof.parquet                                8,610 b
    e3/
      final_model.joblib                       683,300 b
      meta.json                                    426 b
      oof.parquet                                8,610 b
    e4/
      final_model.joblib                       685,684 b
      meta.json                                    442 b
      oof.parquet                                8,610 b
    e5/
      final_model.joblib                       686,292 b
      meta.json                                    468 b
      oof.parquet                                8,610 b
  tables/
    t1.parquet    

Three tables, three sidecars, five experiments. Note `t1.parquet` is byte-for-byte
what it was in section 7 — `t2` and `t3` did not touch it. That immutability is
what makes abandoning a branch free.

## 11. The journal — the run's entire memory

Everything above is on disk in one file. This is what a fresh process, or an
agent after a context reset, reads to know where it is.

In [28]:
j = json.load(open(os.path.join(session.run_dir, "run_journal.json")))
print("keys:", list(j.keys()))
print()
print("task    :", j["task"])
print("baseline:", j["baseline"])
print("best    :", j["best"])
print("stop    :", j["stop"])

keys: ['schema_version', 'task', 'baseline', 'best', 'tables', 'leaderboard', 'log', 'stop']

task    : {'target': 'sold_fast', 'task_type': 'binary', 'metric': 'logloss'}
baseline: {'experiment_id': 'e1', 'cv_score': 0.6903418151196927}
best    : {'experiment_id': 'e4', 'cv_score': 0.46863865262896764, 'artifact_path': 'experiments/e4'}
stop    : {'refusals': 0, 'finalized': False}


In [29]:
# tables[] — one row per derived table, i.e. per feature-building call.
pd.DataFrame(j["tables"])[["id", "path", "source_table", "method", "new_columns"]]

,id,path,source_table,method,new_columns
0,t1,tables/t1.parquet,/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn...,transform,"[month, dayofweek, is_weekend]"
1,t2,tables/t2.parquet,tables/t1.parquet,target,[city_te]
2,t3,tables/t3.parquet,tables/t1.parquet,transform,"[price_per_m2, area_per_room]"


That table *is* the lineage tree: follow `source_table` to walk back to raw. And
for transform rows the journal also stores the `code`, which is the only record
of **what was tried** — an agent resuming later can see the idea, not just the
resulting column names.

In [30]:
print(j["tables"][0]["code"])


import pandas as pd
def transform_features(df):
    df = df.copy()
    ts = pd.to_datetime(df["listed_at"])
    df["month"] = ts.dt.month
    df["dayofweek"] = ts.dt.dayofweek
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
    return df.drop(columns=["listed_at"])



In [31]:
# leaderboard[] — one row per training call.
pd.DataFrame(j["leaderboard"])[
    ["id", "model_type", "source_table", "cv_score", "fold_spread", "vs_baseline"]
]

,id,model_type,source_table,cv_score,fold_spread,vs_baseline
0,e1,baseline,/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn...,0.690342,0.000376,NaN
1,e2,lgbm,/var/folders/yy/fkmybqbn1m97ympktdmxgjxm0000gn...,0.688113,0.041622,0.002229
2,e3,lgbm,tables/t1.parquet,0.497542,0.074699,0.192799
3,e4,lgbm,tables/t2.parquet,0.468639,0.080712,0.221703
4,e5,lgbm,tables/t3.parquet,0.523554,0.040365,0.166788


Two derived pointers, `baseline` and `best`, are recomputed from the whole
leaderboard on every append rather than compared against the previous best. So
they stay correct even if a row is edited or the journal is rebuilt by hand.

**Check yourself:** the journal records facts as a *side effect* of the tools
that produce them — there is no `write_journal` tool the agent calls. Why does
that matter?

> Because a bookkeeping step the agent has to remember is a bookkeeping step the
> agent will skip, or fill in from a faulty memory of what it did. An experiment
> record that is a by-product of running the experiment cannot disagree with the
> experiment. (`journal.py` keeps a separate `append_note` for the *interpretive*
> layer — hypotheses — precisely because that one genuinely does need the agent.)

In [32]:
# The interpretation layer exists in the schema but is not wired to a tool yet.
print("log entries:", len(j["log"]))

log entries: 0


## 12. What the agent structurally cannot do

This is the other half of understanding the harness, and the half that is
invisible when you only read the happy path. Most of the design is about what is
*absent*.

Each of the next five cells tries something a careless or clever model would
try, and shows the refusal. Note that all of them come back as an **error the
model can act on**, not a crash — `_retryable` in `agent.py` converts
`ValueError`, `FileNotFoundError` and `SandboxError` into a `ModelRetry`, so a
bad argument costs one turn instead of discarding every experiment so far.

### 12.1 It cannot read the target in its own code

In [33]:
from tabular_agent.sandbox import SandboxError

cheat = '''
def transform_features(df):
    df = df.copy()
    df["cheat"] = df["sold_fast"]
    return df
'''

try:
    transform_features(session, cheat)
except SandboxError as e:
    print("REFUSED:\n")
    print(str(e)[-460:])

REFUSED:

'sold_fast'

Note: the target column 'sold_fast' is deliberately not in the frame your code receives — it is not available at predict time either. Derive features from the other columns; anything that has to learn from the target belongs in fit_encoding(method='target').


One line of pure pandas, nothing to import — which is exactly why the column is
removed from the frame rather than the code being scanned for the word
`sold_fast`. The error is written to *teach*, because a bare
`KeyError: 'sold_fast'` reads to a model like a typo it should retry.

### 12.2 It cannot fit a model, or anything that learns

In [34]:
sneaky_fit = '''
from sklearn.preprocessing import StandardScaler
def transform_features(df):
    return df
'''

try:
    transform_features(session, sneaky_fit)
except SandboxError as e:
    print("REFUSED:\n")
    print(str(e)[:400])

REFUSED:

'sklearn' is not available in the sandbox: exploration code cannot train models or learn state. -> use train_tabular_model (models) or fit_encoding (encoders) instead.


The sandbox runs an import whitelist: pandas, numpy, scipy and the standard
library. lightgbm, xgboost, catboost and sklearn are blocked, so agent-written
code is *structurally* unable to fit an estimator or learn state.

### 12.3 It cannot smuggle in a stateful transform

In [35]:
qcut_code = '''
import pandas as pd
def transform_features(df):
    df = df.copy()
    df["price_bin"] = pd.qcut(df["price"], 4, labels=False, duplicates="drop")
    return df
'''

try:
    transform_features(session, qcut_code)
except SandboxError as e:
    print("REFUSED:\n")
    print(str(e)[:620])

REFUSED:

these columns depend on which other rows are in the frame: ['price_bin']. The same rows produced different values when transformed as part of a smaller batch, so at predict time — where the batch is whatever arrives, sometimes a single row — they would not match what the model trained on. Causes are pandas operations that read the whole column: pd.qcut, rank, and anything dividing by df[...].mean()/std()/max(). Use fixed thresholds instead (pd.cut with explicit bin edges), or fit_encoding, which learns its parameters once and stores them.


This is the guard that the import block does not cover, and the reason there are
two mechanisms instead of one. `pd.qcut` is pure pandas — nothing to blacklist —
but it takes quantiles **of the batch it is handed**. Since only the code is
stored and replayed, at predict time it would recompute bins from whatever rows
happened to arrive, and the same apartment would land in a different bin
depending on what it was batched with.

Enumerating the guilty pandas calls is unwinnable (any aggregation is one), so
the check tests the *property*: re-run the code over a random subset and reject
any column that disagrees with itself. Row-local arithmetic cannot notice the
difference.

### 12.4 It cannot invent a table path

In [36]:
try:
    train_tabular_model(session, table="tables/t99.parquet")
except ValueError as e:
    print("REFUSED:\n")
    print(e)

REFUSED:

no such table: 'tables/t99.parquet'. Available: None (the raw input table), 'tables/t1.parquet', 'tables/t2.parquet', 'tables/t3.parquet'


The error hands back the menu. The caller is usually a model that mistyped or
hallucinated a path, and the fix it needs is the list of paths that do exist —
not a `FileNotFoundError` three frames later.

### 12.5 It cannot encode the target as a feature

In [37]:
try:
    fit_encoding(session, columns=["sold_fast"], method="target")
except ValueError as e:
    print("REFUSED:", e)

REFUSED: refusing to encode the target column 'sold_fast' as a feature


`fit_encoding` is the one tool that legitimately sees the target, so it is also
the one that needs this specific check.

And one more limit, which has no cell because there is nothing to call: **no
tool returns rows.** Every summary in this notebook is aggregate. The agent has
never seen a single record of this dataset — it has been reasoning from column
names, dtypes and scores the whole way. `explore.py` implements a read-only code
path for exactly this, and it is deliberately **not** registered in `agent.py`
yet.

## 13. It cannot keep going forever

Covered properly in the step-4 notebook; here we just locate where it fires and
what it looks like, since it is part of the map.

The budget lives on the *session*, not the library, because it is a property of
this run. To make the demo deterministic we open a **second session on a fresh
run dir** with a budget of one experiment.

In [38]:
from tabular_agent.harness.stop import StopPolicy

tight = RunSession(
    run_dir=os.path.join(workdir, "tight"),
    data_path=data_path,
    spec=TaskSpec(target="sold_fast", n_splits=4),
    stop_policy=StopPolicy(max_experiments=1),
)
inspect_data(tight)

first = train_tabular_model(tight, model_type="baseline")
print("first call  -> trained:", first["trained"], "| cv_score:", round(first["cv_score"], 4))

second = train_tabular_model(tight, model_type="lgbm")
print("second call -> trained:", second["trained"])
print()
print(json.dumps(second, indent=2))

first call  -> trained: True | cv_score: 0.6903
second call -> trained: False

{
  "trained": false,
  "reason": "this run has used its budget of 1 experiments",
  "n_experiments": 1,
  "since_improvement": 0,
  "next": "finalize_run",
  "note": "The harness is stopping this run: this run has used its budget of 1 experiments. Call finalize_run."
}


The refusal is a **return value, not an exception**. That is the point: being
told the run is over is not a malformed argument, it is a model that has not yet
read the news. Raising would burn a retry and could end the run, throwing away
every experiment already completed.

There are two of these, and they cost the agent very different things:

- **Table exhausted** (`table_patience=3`) — refuses *that table* and redirects
  to feature building. Costs nothing; the next call on a new table goes straight
  through.
- **Run finished** (`patience=10`, `max_experiments=25`) — ends the run, as
  above. Points at `finalize_run`.

Only one of those deserves a hair trigger, which is why the tight limit is the
per-table one and the generous limit is the per-run one.

In [39]:
# Ask the same question as a pure function, no side effects, on the real run.
from tabular_agent.harness import stop as stop_mod

print(json.dumps(
    stop_mod.evaluate(j["leaderboard"], j["task"]["metric"], StopPolicy()), indent=2
))

{
  "should_stop": false,
  "reason": null,
  "n_experiments": 5,
  "since_improvement": 1,
  "patience": 10,
  "experiments_remaining": 20
}


And a third escalation exists behind both: if the harness is asked to train
after refusing `max_refusals=3` times, it **packages the run itself** and then
raises. The reason is that the bundle is the entire deliverable, and a loop that
ends with no bundle has produced nothing.

## 14. Call N — `finalize_run`

Called exactly once. It reads the journal's `best` pointer rather than
re-deriving anything, and writes the two artifacts a human actually wanted.

In [40]:
from tabular_agent.harness.tools import finalize_run

final = finalize_run(session)
print(json.dumps(final, indent=2))

{
  "best_experiment_id": "e4",
  "model_type": "lgbm",
  "metric": "logloss",
  "cv_score": 0.46863865262896764,
  "baseline_score": 0.6903418151196927,
  "n_experiments": 5,
  "bundle_dir": "bundle",
  "report_path": "run_report.md",
  "unmeasured_tables": []
}


In [41]:
tree(session.run_dir)

run/
  run_journal.json                               7,026 b
  run_report.md                                    559 b
  bundle/
    model_0.joblib                             685,924 b
    pipeline.json                                1,137 b
  experiments/
    e1/
      final_model.joblib                           435 b
      meta.json                                    448 b
      oof.parquet                                1,317 b
    e2/
      final_model.joblib                       680,420 b
      meta.json                                    443 b
      oof.parquet                                8,610 b
    e3/
      final_model.joblib                       683,300 b
      meta.json                                    426 b
      oof.parquet                                8,610 b
    e4/
      final_model.joblib                       685,684 b
      meta.json                                    442 b
      oof.parquet                                8,610 b
    e5/
      final_model.

Two new things: `bundle/` and `run_report.md`.

Note **which** experiment won: `e4`, the `t2` model — the one whose extra column
we established in section 9 is pure noise. `finalize_run` reads the journal's
`best` pointer, `best` is the lowest logloss, and `e4` has it. So the run shipped
the model that won on a coin flip.

Is that wrong? No — and it is worth being clear about why. `e4` is a perfectly
good model: it has the date features, and the noise column costs it nothing real.
What would be wrong is *believing* it is better than `e3`. Taking the minimum of
five noisy measurements gives you a number biased low, because you have selected
for lucky folds as much as for good features. The model is fine; the headline
number is optimistic. The report is the place that has to be honest about that,
which is why the next cell is worth reading as a document and not just as output.

`unmeasured_tables` is empty, which is the good case — we trained on every table
we built. When it is not empty it means the winning table has an ancestor that
was never scored, so the score cannot be attributed to the columns on either
side of that link. The run is still *correct* (the model that ships is the one
that scored best), but the *explanation* has a hole, and the report says so
rather than letting the winning table take credit for all of its columns.

In [42]:
manifest = json.load(open(os.path.join(session.run_dir, "bundle", "pipeline.json")))
print("task_type    :", manifest["task_type"])
print("model_paths  :", manifest["model_paths"])
print("weights      :", manifest["weights"])
print("n_features   :", len(manifest["feature_names"]))
print("features     :", manifest["feature_names"])
print("pipeline steps:", [s["type"] for s in manifest["pipeline"]["steps"]])

task_type    : binary
model_paths  : ['model_0.joblib']
weights      : [1.0]
n_features   : 8
features     : ['price', 'area', 'rooms', 'year_built', 'month', 'dayofweek', 'is_weekend', 'city_te']
pipeline steps: ['transform', 'encode']


The bundle inherits the lineage of **whichever table the winning model was
trained on**. That is why the sidecars were kept per table instead of one
run-wide pipeline: the run branched, and only at the end do we learn which
branch to package.

**Check yourself:** if the winning experiment had been `e2` — lgbm on the raw
table — how many pipeline steps would the bundle have?

> Zero. The raw table has no lineage sidecar, so `_load_lineage` returns an
> empty `Pipeline`. The bundle would go straight from raw columns to the model,
> which is correct: that model was trained on nothing but raw columns.

In [43]:
print(open(os.path.join(session.run_dir, "run_report.md")).read())

# Run Report

- **Task type:** binary
- **Selection metric:** logloss
- **CV score:** 0.469 (fold std 0.081)
- **Baseline score:** 0.690
- **Models:** lgbm
- **Features (8):** price, area, rooms, year_built, month, dayofweek, is_weekend, city_te

## Per-fold metrics
- fold 0: 0.3924
- fold 1: 0.3836
- fold 2: 0.5473
- fold 3: 0.5512

## How the winning feature set was built

- `tables/t1.parquet` — transform: month, dayofweek, is_weekend — best 0.4975 over 1 experiment(s)
- `tables/t2.parquet` — target: city_te — best 0.4686 over 1 experiment(s)


Note the report is built from `attribution`, which marks each link in the
winning table's history as measured or not. It reports what the run **entitles
you to conclude** — not just the winning number.

## 15. The proof that the bundle is real

The whole point of storing recipes instead of columns is that the bundle can
take **raw input** — strings, no derived columns, no target — and produce a
prediction. Let us hand it five raw rows.

In [44]:
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(session.run_dir, "bundle"))

raw_rows = df.drop(columns=["sold_fast"]).head(5)
print("what we hand it:", list(raw_rows.columns))
raw_rows

what we hand it: ['price', 'area', 'rooms', 'year_built', 'city', 'listed_at']


,price,area,rooms,year_built,city,listed_at
0,999585.0,111.3,4.0,1990,berlin,2024-03-04
1,454020.0,146.6,6.0,1994,hamburg,2024-02-18
2,1005852.0,130.8,4.0,1976,berlin,2023-04-16
3,289265.0,59.3,1.0,1963,hamburg,2023-11-03
4,516327.0,69.0,2.0,2005,berlin,2023-10-07


In [45]:
preds = predict_bundle(bundle, raw_rows)
print("predictions:", np.round(preds, 4))

predictions: [1.000e-03 1.000e+00 9.984e-01 2.000e-04 7.000e-04]


`listed_at` went in as a string and `city` as a category, and a probability came
out. In between, the stored pipeline re-ran the date transform and applied the
fitted encoder — from the manifest, not from any saved column.

(Those probabilities are near 0 and 1 because these five rows are *training*
rows and `final_model` was refit on all of them. Do not read them as the model
being confident in general — the honest numbers for that are the fold metrics.)

**Check yourself:** the encoder used here was fitted on **all** rows, while the
values the model trained on were out-of-fold. Is that inconsistent?

> No — it is the whole design. Out-of-fold values exist so the *score* is honest
> about what the model knew. At predict time there is no fold and no held-out
> anything, so the best estimate of a city's target mean is the one fitted on
> every row available. Using the out-of-fold values at serve time would be
> throwing away data for no reason.

## 16. The map, in one place

Everything above, as a reference. **Returns** is what the model sees; **writes**
is what persists.

| Call | Writes | Returns |
|---|---|---|
| `inspect_data` | `run_journal.json` → `task` | shape, per-column profile, task type, metric, `non_numeric_columns` |
| `fit_encoding` | `tables/t<N>.parquet` + `.pipeline.json` (fitted encoder); journal `tables[]` | table path, `new_columns` |
| `transform_features` | `tables/t<N>.parquet` + `.pipeline.json` (**the code**); journal `tables[]` | table path, `new_columns` |
| `train_tabular_model` | `experiments/e<N>/` (`meta.json`, `final_model.joblib`, `oof.parquet`); journal `leaderboard[]`, `best`, `baseline` | scores, `columns_by_table`, `vs_source_table`, stop advisory |
| `finalize_run` | `bundle/`, `run_report.md`; journal `stop.finalized` | best experiment, bundle path, report path, `unmeasured_tables` |

And the resulting directory:

```
run_dir/
├── run_journal.json     ← the memory: task, tables[], leaderboard[], best, stop
├── tables/
│   ├── t1.parquet       ← immutable; t2 and t3 are both built ON t1
│   ├── t1.pipeline.json ←   how to rebuild its columns from raw
│   └── ...
├── experiments/
│   └── e1/  meta.json, final_model.joblib, oof.parquet
├── bundle/              ← written once, by finalize_run
│   ├── model_0.joblib
│   └── pipeline.json
└── run_report.md
```

The loop is: **build a table → train on it → read the feedback → build another
table.** Five tools, one folder.

### Where the map has a blank spot

Having driven all five tools by hand, the gap is easier to see than it was in
the abstract.

The cycle a human runs is: inspect → hypothesise → build → train → **score
(look at out-of-fold predictions)** → hypothesise again. Five of those six arcs
exist above. The missing one is `score → hypothesise`: `oof.parquet` is written
on every training call and read by nothing, so when the agent decides what to
build next it is working from column names and one scalar per experiment.

That is why it reaches for whatever sounds plausible rather than for whatever
the model is currently bad at. Closing that arc is the next slice, and it lands
on the constraint from section 12: doing it means showing the agent something
row-level for the first time.

There is a second blank spot, and sections 9 and 14 walked straight into it.
`cv_score` is both **the number the agent optimises** and **the number the report
headlines**. Five experiments were enough for that to pick a noise column as the
winner. Nothing in this run computes a score on data that no selection decision
ever touched — `build_report` accepts a `holdout_score` argument and no caller
ever passes one. That matters more once the agent can see residuals, because then
it is choosing features using out-of-fold errors, and the out-of-fold score stops
being an innocent bystander.

## 17. Now let the model do it

Everything above was a hand-driven sequence of the same five functions the agent
gets. `build_agent` adds nothing but a system prompt and the retry wrapper.

This cell costs money and needs a key, so it is the last thing in the notebook
and it is left for you to run deliberately.

In [ ]:
from tabular_agent.harness.agent import build_agent, gateway_model, load_api_key

# Either a provider string ("anthropic:claude-opus-5") or a gateway model.
model = gateway_model("tensorx/deepseek-v4-flash", api_key=load_api_key())

live = RunSession(
    run_dir=os.path.join(workdir, "agent_run"),
    data_path=data_path,
    spec=TaskSpec(target="sold_fast", n_splits=4),
)

result = await build_agent(model).run(
    "Build the best model you can for this dataset, then finalize the run.",
    deps=agent_mod.Deps(session=live),
)
print(result.output)

In [ ]:
# The same x-ray, on a run nobody drove by hand.
tree(live.run_dir)

lj = json.load(open(os.path.join(live.run_dir, "run_journal.json")))
pd.DataFrame(lj["leaderboard"])[
    ["id", "model_type", "source_table", "cv_score", "vs_baseline"]
]

Compare that leaderboard against ours. The interesting question is not whether
it beat us — it is **which of the five tools it reached for, in what order, and
what it never tried.** That is the search behaviour the next slice is meant to
improve, and this notebook is the baseline you are comparing against.